# Attention weights of the reference encoder: what they actually measure

**Model:** Tacotron2-VAE with attention *pooling* (`verbo_v4_f1`, checkpoint `epoch_9500`) · **Corpus:** VERBO f1 (98 utterances, 7 emotions)

## Why the analyses on the site do not make sense

All the published analyses (overlay on the time axis, "~0.74 s windows", entropy as temporal spread,
correlation with energy per time window) assume that the index $t$ of $\alpha_t$ runs over **time**. This notebook
shows that this premise is **false** for the models trained in this repository:

`ReferenceEncoder._cnn_forward` does `inputs.contiguous().view(B, 1, -1, n_mels)` on a mel with layout **(B, 80, T)**.
`view` does not transpose; it reinterprets the memory. The result is that the "image" seen by the convolutions is not (time × mel),
and each GRU step ends up covering a **band of mel channels across the WHOLE sentence**, not a time window:

- step 0 → mel channels ≈ 0–19 (low) · all time instants
- step 1 → channels ≈ 20–39 · all time instants
- … · last step → the highest channels

In other words: **α is a SPECTRAL importance profile, not a temporal one.** This changes how everything reads: the "recency bias"
is weight on the high bands; "sadness concentrated at the end" is sadness that depends on the high bands; and the weak correlation
with energy per time window was to be expected: wrong axis.

## What this notebook does

1. **Proof of the layout** (Section 1): exact arithmetic of the `view` + receptive field by gradient. Does not depend on a checkpoint.
2. **Spectral profile of α** over the whole corpus (Section 3): 98 utterances, not 7.
3. **Causal validation by occlusion on BOTH axes** (Section 4): mask time windows vs. mel bands and measure the effect
   on the style embedding $\mu$. If α measures influence, it should predict the effect of occluding **bands** and ignore that of **time**.
4. **Gradient saliency** (Section 5): an independent attribution to compare against α.
5. **Emotions with n=98 and a statistical test** (Section 6): per-emotion profiles on the right axis.

> Run on the machine that has the checkpoint (`experiments/tacotron2-vae/verbo_v4_f1/checkpoints/epoch_9500`), or export
> `ATTN_CHECKPOINT=/path/to/checkpoint`. `ATTN_SMOKE_TEST=1` runs with random weights (only to test the code).

In [ ]:
import os, sys, json, math
from pathlib import Path
from collections import OrderedDict

if Path.cwd().name == 'notebooks':
    os.chdir(Path.cwd().parent)
ROOT = Path.cwd()
for p in ['src', 'src/models/tacotron2_vae', 'src/training/training-tacotron2-vae']:
    sys.path.insert(0, str(ROOT / p))

import numpy as np
import pandas as pd
import torch
import torchaudio
import librosa
import matplotlib.pyplot as plt
from scipy import stats

from layers import TacotronSTFT
from hparams import Tacotron2VAEHparams
from model import load_tacotron2_vae_model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(0); np.random.seed(0)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.25,
                     'axes.spines.top': False, 'axes.spines.right': False})
MEL_FLOOR = float(np.log(1e-5))   # silence floor of the pipeline (dynamic_range_compression)
RESULTS = {}
print('device:', device)

In [ ]:
# ---------- model loading (with path discovery) ----------
CHECKPOINT = os.environ.get('ATTN_CHECKPOINT', '')
SMOKE = os.environ.get('ATTN_SMOKE_TEST') == '1'

EXP_CANDIDATES = [ROOT / 'experiments/tacotron2-vae/verbo_v4_f1',
                  ROOT / 'experiments/drive/tacotron2-vae/verbo_v4_f1']
DATA_CANDIDATES = [ROOT / 'data/raw/verbo/Audios/f1',
                   ROOT / 'data/raw/VERBO-Dataset/Audios/f1']

EXPERIMENT = next(p for p in EXP_CANDIDATES if (p / 'hparams.json').exists())
DATA_DIR = next(p for p in DATA_CANDIDATES if p.exists() and any(p.glob('*.wav')))

hparams = Tacotron2VAEHparams.from_dict(json.load(open(EXPERIMENT / 'hparams.json')))
model = load_tacotron2_vae_model(hparams, device=device)

ck_path = Path(CHECKPOINT) if CHECKPOINT else None
if ck_path is None:
    ckdir = EXPERIMENT / 'checkpoints'
    cks = sorted(ckdir.glob('epoch_*'), key=lambda p: int(p.name.split('_')[1])) if ckdir.exists() else []
    ck_path = cks[-1] if cks else None

if ck_path is not None and ck_path.exists():
    ck = torch.load(ck_path, map_location='cpu', weights_only=False)
    sd = OrderedDict((k[7:] if k.startswith('module.') else k, v)
                     for k, v in ck.get('state_dict', ck).items())
    assert any('.ref_encoder.attn.' in k for k in sd), (
        f'{ck_path} does NOT have attention pooling (it is a baseline); point ATTN_CHECKPOINT '
        'to the model with attention, e.g.: verbo_v4_f1/checkpoints/epoch_9500')
    model.load_state_dict(sd)
    print(f'checkpoint: {ck_path}  (iteration {ck.get("iteration", "?")})')
elif SMOKE:
    print('*' * 70)
    print('SMOKE TEST: RANDOM WEIGHTS - meaningless numbers, only validates the code')
    print('*' * 70)
else:
    raise FileNotFoundError('Checkpoint with attention not found; export ATTN_CHECKPOINT '
                            'or run on the machine with verbo_v4_f1/checkpoints/epoch_9500.')

model.eval()
ref_enc = model.vae_gst.ref_encoder
N_MELS, HOP, SR = hparams.n_mel_channels, hparams.hop_length, hparams.sampling_rate
MEL_HZ = librosa.mel_frequencies(n_mels=N_MELS, fmin=hparams.mel_fmin, fmax=hparams.mel_fmax)
print(f'latent L={hparams.z_latent_dim} · GRU={hparams.ref_enc_gru_size} · data: {DATA_DIR}')

## 1 · Proof of the layout: what each GRU step covers

The nominal block of step $w$ is rows $[64w,\,64w{+}64)$ of the post-`view` image. Each row $r$ contains the linear
indices $[80r,\,80r{+}80)$ of the memory of the mel (80, T), whose linear index is $p = m\cdot T + t$. Arithmetic alone is enough to
map each cell (channel $m$, time $t$) to its step, without running the model.

In [ ]:
def step_of_cell(T, n_steps):
    '''(80, T) matrix with the index of the GRU step that "owns" each mel cell.'''
    m_idx, t_idx = np.meshgrid(np.arange(N_MELS), np.arange(T), indexing='ij')
    flat = m_idx * T + t_idx                # linear index in the (80, T) memory
    row = flat // N_MELS                    # row of the post-view image
    return np.minimum(row // 64, n_steps - 1)

def n_steps_for(T):
    '''Output height of the 6 stride-2 convs (k=3, pad=1): floor((H-1)/2)+1, 6x.'''
    h = T
    for _ in range(6):
        h = (h - 1) // 2 + 1
    return h

def band_ranges(T, n_steps):
    '''Range of mel channels [lo, hi] covered by the nominal block of each step.'''
    out = []
    for w in range(n_steps):
        lo = min(N_MELS - 1, (64 * w * N_MELS) // T)
        hi = min(N_MELS - 1, (64 * (w + 1) * N_MELS - 1) // T)
        out.append((lo, hi))
    return out

T_demo = 258                                    # typical utterance of ~3 s
ns = n_steps_for(T_demo)
owner = step_of_cell(T_demo, ns)

fig, ax = plt.subplots(figsize=(9, 3.2))
im = ax.imshow(owner, aspect='auto', origin='lower', cmap='tab10',
               extent=[0, T_demo * HOP / SR, 0, N_MELS])
ax.set(xlabel='time (s)', ylabel='mel channel',
       title=f'Owner of each mel cell (T={T_demo}, {ns} steps) — HORIZONTAL stripes = frequency bands')
plt.colorbar(im, ax=ax, label='GRU step')
plt.tight_layout(); plt.show()

for w, (lo, hi) in enumerate(band_ranges(T_demo, ns)):
    print(f'step {w}: mel channels {lo:2d}-{hi:2d}  ({MEL_HZ[lo]:5.0f}-{MEL_HZ[hi]:5.0f} Hz) — the WHOLE sentence in time')

If α were temporal, the map above would have **vertical** stripes. The stripes are **horizontal**: each weight governs a
frequency band of the whole sentence. Confirmation with the real model (receptive field by gradient; the convolutions
blur ±1 step at the edges, but the dominant support is the band):

In [ ]:
# gradient of the GRU input at step w with respect to the mel -> real support (with the convs' overlap)
mel_probe = torch.randn(1, N_MELS, T_demo, device=device) * 0.1 + MEL_FLOOR
mel_probe.requires_grad_(True)
gru_in = ref_enc._cnn_forward(mel_probe)        # (1, T', 256)
ns_real = gru_in.size(1)
fig, axes = plt.subplots(1, min(3, ns_real), figsize=(13, 2.8), sharey=True)
for ax, w in zip(np.atleast_1d(axes), [0, ns_real // 2, ns_real - 1]):
    g = torch.autograd.grad(gru_in[0, w].pow(2).sum(), mel_probe, retain_graph=True)[0]
    ax.imshow(g[0].abs().cpu().numpy(), aspect='auto', origin='lower', cmap='magma')
    ax.set(title=f'|∂ step {w} / ∂ mel|', xlabel='time (frames)')
np.atleast_1d(axes)[0].set_ylabel('mel channel')
plt.suptitle('Real receptive field of each step: horizontal stripes (bands), not vertical (time)', y=1.05)
plt.tight_layout(); plt.show()
print(f'real steps of the conv stack for T={T_demo}: {ns_real} (formula: {n_steps_for(T_demo)})')

## 2 · Corpus and extraction

We extract α and the style embedding μ for the **98 utterances** of speaker f1 (14 per emotion) and build a table per
(utterance, step) with the corresponding mel band. Since the number of steps depends on the duration ($T' = $ post-conv height), the
bands have a different resolution per utterance; that is why everything below uses the **central channel of the band** as the common axis.

In [ ]:
EMO = {'neu': 'neutral', 'ale': 'joy', 'rai': 'anger', 'tri': 'sadness',
       'med': 'fear', 'sur': 'surprise', 'des': 'disgust'}
stft = TacotronSTFT(filter_length=hparams.filter_length, hop_length=HOP, win_length=hparams.win_length,
                    n_mel_channels=N_MELS, sampling_rate=SR, mel_fmin=hparams.mel_fmin, mel_fmax=hparams.mel_fmax)

utts, rows = [], []
MAX_UTTS = int(os.environ.get('ATTN_MAX_UTTS', '0')) or None
for wav_path in sorted(DATA_DIR.glob('*.wav'))[:MAX_UTTS]:
    codee = wav_path.name.split('-')[0]
    if codee not in EMO:
        continue
    wave, sr = torchaudio.load(str(wav_path))
    wave = wave.mean(dim=0, keepdim=True)
    if sr != SR:
        wave = torchaudio.functional.resample(wave, sr, SR)
    wave = wave.clamp(-1, 1)
    with torch.no_grad():
        mel = stft.mel_spectrogram(wave).to(device)                 # (1, 80, T)
        enc, alpha = ref_enc.encode_with_attention(mel)
        _, mu, _, _ = model.vae_gst._vae_from_enc(enc)
    a = alpha[0].cpu().numpy(); T = mel.size(2)
    bands = band_ranges(T, len(a))
    u = dict(utt=wav_path.stem, emotion=EMO[codee], mel=mel, alpha=a,
             mu=mu[0].cpu().numpy(), T=T, bands=bands, dur=wave.size(1) / SR)
    utts.append(u)
    for w, (aw, (lo, hi)) in enumerate(zip(a, bands)):
        band_e = mel[0, lo:hi + 1].mean().item()                    # mean log-mel energy of the band
        rows.append(dict(utt=u['utt'], emotion=u['emotion'], w=w, n_w=len(a),
                         alpha=float(aw), ch_lo=lo, ch_hi=hi, ch_c=(lo + hi) / 2,
                         hz_c=float(MEL_HZ[(lo + hi) // 2]), band_energy=band_e))
df = pd.DataFrame(rows)
print(f'{len(utts)} utterances · {len(df)} (utterance, band) pairs · steps/utterance: '
      f'{df.groupby("utt")["w"].max().add(1).min()}-{df.groupby("utt")["w"].max().add(1).max()}')
df.head()

## 3 · The spectral profile of the attention

The corrected question: **which frequency bands does the encoder consider informative for style?**
And is α just a per-band energy detector, or does it carry more than that?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
# mean alpha profile by central channel (all utterances)
df['alpha_norm'] = df['alpha'] * df['n_w']            # 1.0 = uniform, independent of the number of steps
bins = pd.cut(df['ch_c'], bins=np.arange(0, 81, 8))
prof = df.groupby(bins, observed=True)['alpha_norm'].agg(['median', lambda s: s.quantile(.25), lambda s: s.quantile(.75)])
x = [iv.mid for iv in prof.index]
axes[0].axhline(1.0, color='gray', ls='--', lw=1, label='uniform')
axes[0].plot(x, prof['median'], 'o-', color='#2563eb', label='median')
axes[0].fill_between(x, prof.iloc[:, 1], prof.iloc[:, 2], alpha=.2, color='#2563eb')
axes[0].set(xlabel='central mel channel of the band', ylabel='α / uniform',
            title='Mean spectral profile of the attention (98 utterances)')
sec = axes[0].secondary_xaxis('top', functions=(lambda c: np.interp(c, np.arange(80), MEL_HZ / 1000),
                                                lambda f: np.interp(f, MEL_HZ / 1000, np.arange(80))))
sec.set_xlabel('frequency (kHz)')
axes[0].legend()

# alpha vs band energy (correlation on the RIGHT axis, within each utterance)
rho_e = df.groupby('utt').apply(lambda g: stats.spearmanr(g['alpha'], g['band_energy'])[0]
                                if g['w'].nunique() > 2 else np.nan, include_groups=False).dropna()
axes[1].hist(rho_e, bins=21, color='#2563eb', alpha=.8)
axes[1].axvline(rho_e.median(), color='crimson', label=f'median ρ = {rho_e.median():.2f}')
axes[1].set(xlabel='Spearman ρ(α, band energy) per utterance', ylabel='no. of utterances',
            title='Does α follow the spectral energy?')
axes[1].legend(); plt.tight_layout(); plt.show()
RESULTS['rho_alpha_energia_banda'] = float(rho_e.median())
print(f"median of ρ(α, band energy) = {rho_e.median():.3f}  "
      f"(Wilcoxon vs 0: p = {stats.wilcoxon(rho_e).pvalue:.2e})")

## 4 · Causal validation: occlusion on both axes

The decisive test. For each utterance we mask (silence floor, `log(1e-5)`):

- **bands**: the mel channels of each step's nominal block (the spectral hypothesis);
- **time**: windows of $T/T'$ frames (the site's (wrong) interpretation);

and we measure $\Delta\mu = \lVert \mu_{\text{masked}} - \mu_{\text{full}} \rVert$. If α is a measure of influence,
$\rho(\alpha, \Delta\mu_{\text{band}})$ should be high and $\rho(\alpha, \Delta\mu_{\text{time}})$ should be ~0.

In [ ]:
def mu_of(mel):
    with torch.no_grad():
        enc, _ = ref_enc.encode_with_attention(mel)
        _, mu, _, _ = model.vae_gst._vae_from_enc(enc)
    return mu[0].cpu().numpy()

rho_band, rho_time, pooled = [], [], []
for u in utts:
    mel, a, T, n = u['mel'], u['alpha'], u['T'], len(u['alpha'])
    if n <= 2:
        continue
    d_band, d_time = [], []
    tw = T / n
    for w, (lo, hi) in enumerate(u['bands']):
        mb = mel.clone(); mb[0, lo:hi + 1, :] = MEL_FLOOR            # occlude the BAND
        d_band.append(np.linalg.norm(mu_of(mb) - u['mu']))
        t0, t1 = int(w * tw), int((w + 1) * tw)
        mt = mel.clone(); mt[0, :, t0:t1] = MEL_FLOOR                # occlude the TIME WINDOW
        d_time.append(np.linalg.norm(mu_of(mt) - u['mu']))
    rho_band.append(stats.spearmanr(a, d_band)[0])
    rho_time.append(stats.spearmanr(a, d_time)[0])
    db = np.array(d_band)
    pooled += [(ai, di / db.max()) for ai, di in zip(a * n, db)]

rho_band, rho_time = np.array(rho_band), np.array(rho_time)
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
axes[0].boxplot([rho_band, rho_time], tick_labels=['BAND occlusion', 'TIME occlusion'], widths=.5)
axes[0].axhline(0, color='gray', lw=1)
axes[0].set(ylabel='Spearman ρ(α, Δμ) per utterance', title='Which occlusion\'s causal effect does α predict?')
px, py = np.array(pooled).T
axes[1].scatter(px, py, s=8, alpha=.35, color='#2563eb')
axes[1].set(xlabel='α / uniform', ylabel='band Δμ (normalized per utterance)',
            title=f'Pooled: ρ = {stats.spearmanr(px, py)[0]:.2f}')
plt.tight_layout(); plt.show()

RESULTS['rho_occl_banda'] = float(np.median(rho_band))
RESULTS['rho_occl_tempo'] = float(np.median(rho_time))
print(f'median ρ(α, Δμ_band) = {np.median(rho_band):.3f}   '
      f'median ρ(α, Δμ_time) = {np.median(rho_time):.3f}')
print(f'Paired Wilcoxon band vs time: p = {stats.wilcoxon(rho_band, rho_time).pvalue:.2e}')

**How to read it:** a high ρ for band occlusion = α is a *causally validated* attribution map: removing the band
it points to changes the style in proportion to the weight. A ρ of ~0 for time occlusion confirms that reading α as a timeline
(as the site does) has no causal support.

## 5 · Gradient saliency: an independent attribution

$\lVert \partial \mu / \partial \text{mel} \rVert$ aggregated per band is a second importance map, computed without
using α. If the two agree, the spectral reading becomes even more solid.

In [ ]:
def saliency(mel):
    m = mel.clone().requires_grad_(True)
    with torch.backends.cudnn.flags(enabled=False):   # RNN backward in eval mode requires cudnn off
        enc, _ = ref_enc.encode_with_attention(m)
        _, mu, _, _ = model.vae_gst._vae_from_enc(enc)
        mu.pow(2).sum().backward()
    return m.grad[0].abs().cpu().numpy()          # (80, T)

rho_grad = []
show = [u for e in ['neutral', 'sadness', 'anger'] for u in utts if u['emotion'] == e][:3]
fig, axes = plt.subplots(1, len(show), figsize=(13, 3.0))
for u in utts:
    sal = saliency(u['mel'])
    g_band = [sal[lo:hi + 1].mean() for lo, hi in u['bands']]
    if len(g_band) > 2:
        rho_grad.append(stats.spearmanr(u['alpha'], g_band)[0])
    if u in show:
        ax = np.atleast_1d(axes)[show.index(u)]
        ax.imshow(sal, aspect='auto', origin='lower', cmap='magma')
        for lo, hi in u['bands'][:-1]:
            ax.axhline(hi + .5, color='cyan', lw=.6, ls=':')
        ax.set(title=f"{u['utt']} ({u['emotion']})", xlabel='time (frames)')
np.atleast_1d(axes)[0].set_ylabel('mel channel')
plt.suptitle('|∂μ/∂mel|: the saliency is organized in HORIZONTAL STRIPES (bands; cyan lines = step boundaries)', y=1.06)
plt.tight_layout(); plt.show()

rho_grad = np.array(rho_grad)
RESULTS['rho_grad'] = float(np.median(rho_grad))
print(f'median ρ(α, band saliency) = {np.median(rho_grad):.3f}  '
      f'(Wilcoxon vs 0: p = {stats.wilcoxon(rho_grad).pvalue:.2e})')

## 6 · Emotions, with the whole corpus and the right axis

A reinterpretation of the site's finding ("sadness concentrates attention at the end") with n=98: spectral profiles of α per emotion.
"End" = **high bands**. We also compare with the per-band energy profile, to separate "α follows the emotion"
from "α follows the energy, which changes with the emotion".

In [ ]:
# project each utterance's alpha onto a fixed grid of 80 channels
def to_grid(u, values):
    g = np.zeros(N_MELS)
    for (lo, hi), v in zip(u['bands'], values):
        g[lo:hi + 1] = v
    return g

grid_a = {e: [] for e in EMO.values()}
ent = []
for u in utts:
    n = len(u['alpha'])
    grid_a[u['emotion']].append(to_grid(u, u['alpha'] * n))
    if n > 1:
        ent.append(dict(emotion=u['emotion'],
                        H=float(-(u['alpha'] * np.log(u['alpha'] + 1e-12)).sum() / np.log(n))))
ent = pd.DataFrame(ent)

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
cmap = plt.get_cmap('tab10')
for i, (e, curves) in enumerate(grid_a.items()):
    axes[0].plot(np.arange(N_MELS), np.mean(curves, axis=0), color=cmap(i), label=e, lw=1.8)
axes[0].axhline(1.0, color='gray', ls='--', lw=1)
axes[0].set(xlabel='mel channel', ylabel='α / uniform', title='Mean spectral profile of α per emotion (n=14 each)')
axes[0].legend(ncol=2, fontsize=8)

order = ent.groupby('emotion')['H'].median().sort_values().index
axes[1].boxplot([ent.loc[ent.emotion == e, 'H'] for e in order], tick_labels=list(order), widths=.6)
kw = stats.kruskal(*[ent.loc[ent.emotion == e, 'H'] for e in order])
axes[1].set(ylabel='normalized entropy of α', title=f'Entropy per emotion — Kruskal-Wallis p = {kw.pvalue:.3g}')
axes[1].tick_params(axis='x', rotation=30)
plt.tight_layout(); plt.show()
RESULTS['kruskal_entropia_p'] = float(kw.pvalue)
print(ent.groupby('emotion')['H'].agg(['median', 'std']).round(3).sort_values('median'))

## 7 · Conclusions

**The useful meaning of the reference encoder's attention weights is spectral, not temporal:**

1. **Layout (Section 1, architectural fact):** the `view` without a transpose makes each GRU step cover a band of mel channels
   of the whole sentence. Every temporal reading of α (site: "0.74 s windows", "recency bias", "attention at the end of the
   sentence") is on the wrong axis. The "recency bias" is, in fact, weight on the **high bands**.
2. **α as a causally validated spectral attribution (Sections 4–5):** occluding the band α points to moves the embedding μ
   in proportion to the weight (high median ρ), occluding time windows bears no relation to α, and gradient
   saliency agrees with α. In other words: **α says which frequency ranges of the reference build the style vector**:
   an "importance equalizer" over the speaker's timbre/register.
3. **Emotions (Section 6):** the per-emotion profiles should be read as spectral signatures (e.g., high-arousal
   emotions spread importance across the bands; low arousal concentrates it in specific bands), with n=98 and a statistical
   test, not 7 numbers.

**Practical consequences:**
- The site/report text about "~0.74 s windows" and the figures with α overlaid on the time axis need to be
  rewritten as spectral profiles (α vs. mel channel/Hz).
- If the original intent was truly *temporal* attention, the fix is one line,
  `inputs.transpose(1, 2).contiguous().view(B, 1, -1, n_mels)` in `ReferenceEncoder._cnn_forward`, but it requires
  **retraining** the models; the current checkpoints should be interpreted as spectral attention.
- The interpretability is still real, and even more interesting: the model learned on its own to weight frequency
  bands to extract style, and the weighting is causally faithful.

In [ ]:
print('=' * 64)
print('AUTOMATIC SUMMARY')
print('=' * 64)
labels = {
    'rho_alpha_energia_banda': 'median ρ (α, band energy)                ',
    'rho_occl_banda':          'median ρ (α, Δμ BAND occlusion)  [high = α is causal]',
    'rho_occl_tempo':          'median ρ (α, Δμ TIME occlusion)  [~0 = temporal reading invalid]',
    'rho_grad':                'median ρ (α, gradient saliency)          ',
    'kruskal_entropia_p':      'Kruskal-Wallis entropy×emotion (p)       ',
}
for k, v in RESULTS.items():
    print(f'  {labels.get(k, k)} = {v:.4g}')
if os.environ.get('ATTN_SMOKE_TEST') == '1':
    print('\n  [SMOKE TEST: random weights -> these numbers mean NOTHING]')